# Functions for data collection notebook

packages used: 
- pandas (version)
- exa_py (version 2.16.0)
- google-genai (version 2.12.1)
- pydantic (version 2.13.4)
- typing (version 3.7.4.3)
- time (version)

## 1- Imports and versions of the packages

In [ ]:
import pandas as pd

from exa_py import Exa

from google import genai
from pydantic import BaseModel, Field
from typing import List, Optional

from time import sleep

## 2- API keys and initialisations

In [ ]:
# defining the api keys

# search api
EXA_API_KEY = "key"

# llm
GEMINI_API_KEY = "key "


In [13]:
# initialisation of exa
exa = Exa(api_key=EXA_API_KEY)

# initialisation of gemini
client = genai.Client(api_key=GEMINI_API_KEY)

In [14]:
# structure for the llm output

class Violations(BaseModel):
    n_violations: int = Field(
        description="Number of human rights violations"
    )

    violation_sentences: List[str] = Field(
        description="The exact extracts from the text that describe human rights violations, each describing a different one, including if possible the country and details of the violation. There should be one extract per violation"
    )
    author: Optional[str] = Field(
        default=None,
        description="The author of the document, if explicitly stated."
    )
    date: Optional[str] = Field(
        default=None,
        description="The publication or document date, if explicitly stated."
    )
    title: Optional[str] = Field(
        default=None,
        description="Title of the document, if explicitely stated"
    )

## 3- Function

In [ ]:
def collection(query, n_answers=20, url_seen=[]):
    '''
    input: 
    query: str, keywords used for the search
    n_answers: int, number of documents to return
    url_seen: list, list of urls that have already been treated

    output: 
    pandas dataframe with details on the documents analysed
    '''

    # step 0: creating an empty dataframe to store the results
    data = pd.DataFrame(columns=["url", "title", "authors", "date", "extract", "country", "theme"])

    # step 1: searching for documents on internet
    print("-"*20)
    print("searching internet with query: ", query)
    
    results = exa.search(
        query=query,
        num_results=n_answers,
        user_location="GB",
        exclude_domains=["wikipedia.org"],
        category="news"
    )
    print("done")

    # step 2: getting a list of the new urls collected

    urls = []
    for i in range(len(results.results)):
        url = results.results[i].url

        if url not in url_seen: # checking against the urls that have already been treated
            urls.append(url)
    
    print("Number of new urls collected: ", len(urls))
    print("-"*20)

    
    # step 3: for each url, get the content, pass it through the LLM
    
    i = 1 # initialising counter for printing

    for url in urls:

        print(f"Analysing url {i} of {len(urls)}")

        # get the content of the url
        content = exa.get_contents(
            urls=[url],
            text=True
        )
        text = content.results[0].text

        print("content collected")

        # using the llm to extract the information

        interaction = client.interactions.create(
            model="gemini-3.5-flash",
            input=f"Return the detailed instances of human rights violation against a specific person with albinism that happened in Africa: {text}",
            response_format={
                "type": "text",
                "mime_type": "application/json",
                "schema": Violations.model_json_schema()
            }
        )

        print("gemini successful")

        llm_output = interaction.output_text
        output = Violations.model_validate_json(llm_output)

        # adding the new extracts to the results dataframe
        for extract in output.violation_sentences:
            new_row = pd.DataFrame({
                "url":[url], 
                "title":[output.title], 
                "authors":[output.author],
                "date":[output.date],
                "extract":[extract],
                "country":None,
                "theme":None

            })
            data = pd.concat([data, new_row], ignore_index=True)
        
        print(f"{output.n_violations} were added to the dataframe")

        i += 1

        if i > 10:
            break

        # sleeping for 1 minute to make sure to not exceed the max number of requests per minute
        sleep(60)

    return data


    
    
